In [10]:
import numpy as np

SIGMOID = 'sigmoid'
RELU = 'relu'
TANH = 'tanh'

class MLP:
    def __init__(self, layers, activation_list, learning_rate=0.01):
        self.layers = layers
        self.activation_list = activation_list
        self.lr = learning_rate
        self.weight_list = []
        self.biase_list = []

        self._init_weights()

    def _init_weights(self):
        for i in range(len(self.layers) - 1):
            w = np.random.randn(self.layers[i], self.layers[i+1]) * 0.01
            b = np.zeros((1, self.layers[i+1]))
            self.weight_list.append(w)
            self.biase_list.append(b)

    def _activation(self, Z, func):
        if func == SIGMOID:
            return 1 / (1 + np.exp(-Z))
        elif func == RELU:
            return np.maximum(0, Z)
        elif func == TANH:
            return np.tanh(Z)
        else:
            raise ValueError("Activation not allowed")

    def _activation_derivative(self, A, func):
        if func == SIGMOID:
            return A * (1 - A)
        elif func == RELU:
            return (A > 0).astype(float)
        elif func == TANH:
            return 1 - A**2

    def forward(self, X):
        f_pred = X
        f_pred_turn = [f_pred]

        for i in range(len(self.weight_list)):
            Z = f_pred @ self.weight_list[i] + self.biase_list[i]
            f_pred = self._activation(Z, self.activation_list[i])
            f_pred_turn.append(f_pred)

        return f_pred, f_pred_turn

    def compute_loss(self, y_pred, y_true):
        return np.mean((y_pred - y_true) ** 2)

    def backward(self, y_pred_turn, y_true):
        grads_w = []
        grads_b = []

        last_pred = y_pred_turn[-1]
        d_last_pred = (last_pred - y_true)

        for i in reversed(range(len(self.weight_list))):
            A = y_pred_turn[i]
            A_curr = y_pred_turn[i + 1]

            dZ = d_last_pred * self._activation_derivative(A_curr, self.activation_list[i])
            dW = A.T @ dZ / len(y_true)
            dB = np.mean(dZ, axis=0, keepdims=True)

            d_last_pred = dZ @ self.weight_list[i].T

            grads_w.insert(0, dW)
            grads_b.insert(0, dB)

        return grads_w, grads_b

    def update(self, grads_w, grads_b):
        for i in range(len(self.weight_list)):
            self.weight_list[i] -= self.lr * grads_w[i]
            self.biase_list[i] -= self.lr * grads_b[i]

    def fit(self, X, y, epochs=1000):
        for epoch in range(epochs):
            y_pred, y_pred_turn = self.forward(X)
            loss = self.compute_loss(y_pred, y)

            grads_w, grads_b = self.backward(y_pred_turn, y)
            self.update(grads_w, grads_b)

            if epoch % 100 == 0:
                print(f"Epoch {epoch} - Loss: {loss:.4f}")

In [11]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

X, y = make_circles(n_samples=1000, noise=0.1, factor=0.5)
y = y.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
mlp = MLP(
    layers=[2, 10, 5, 1],
    activation_list=["relu", "relu", "sigmoid"],
    learning_rate=0.01
)

mlp.fit(X_train, y_train, epochs=1000)

Epoch 0 - Loss: 0.2564
Epoch 100 - Loss: 0.2560
Epoch 200 - Loss: 0.2556
Epoch 300 - Loss: 0.2552
Epoch 400 - Loss: 0.2549
Epoch 500 - Loss: 0.2545
Epoch 600 - Loss: 0.2542
Epoch 700 - Loss: 0.2538
Epoch 800 - Loss: 0.2534
Epoch 900 - Loss: 0.2531
